In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import rasterio
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

In [ ]:
mask_path = "/content/drive/MyDrive/cosesmic/layers/raster_class/low_lsi_mask.tif"

In [ ]:
mask_path_random="/content/drive/MyDrive/cosesmic/layers/random/slope_mask_15deg.tif"

In [ ]:
n_points = 15000

In [ ]:
output_path = "/content/drive/MyDrive/cosesmic/layers/pseudo_absence_points.gpkg"

In [ ]:
output_path_random="/content/drive/MyDrive/cosesmic/layers/random/pseudi_absence_points_random"

In [ ]:
!ls /content/drive/MyDrive/cosesmic/layers

 aligned_raw			 non_landslide_points_500.gpkg
 chrips_clipped			 non_landslide_points.gpkg
 classified_rasters		 non_landslide_points_with_lsi_500.csv
 distances			 non_landslide_points_with_lsi.csv
 final_pseudoabsence_area.gpkg	 output_fr_rasters
 FR_rasters_fixed		 output_fr_rasters1
 fr_vector			 output_fr_rasters2
 landslide_buffer_500.gpkg	 output_fr_rasters_raw
 landslide_buffer.gpkg		 output_fr_rasters_raw1
 landslide_points.geojson	 output_weight_rasters
 landslide_points_with_lsi.csv	 output_weight_rasters_aligned
 landslide_positive.gpkg	 pseudo_absence_filtered.gpkg
 landslide_samples.gpkg		 pseudo_absence_points.gpkg
 low_lsi_area.gpkg		 pseudo_absense
 LSI_classes1.tif		'pseudo_absense (1)'
 LSI_classes1.tif.aux.xml	 random
 LSI_classes2.tif		 random_non_landslide.gpkg
 LSI_classes.tif		 raster_class
 LSI_classes.tif.aux.xml	 rasters
 LSI_map.tif			 safe_area_500.gpkg
 LSI_map.tif.aux.xml		 safe_area.gpkg
'New folder'			'unique vlue 1'
'New folder (2)'		 Uttarakhand.geojs

In [ ]:
# READ MASK

with rasterio.open(mask_path) as src:
    mask = src.read(1)
    transform = src.transform
    crs = src.crs


In [ ]:
with rasterio.open(mask_path_random) as src:
    mask = src.read(1)
    transform = src.transform
    crs = src.crs

In [ ]:
# VALID PIXELS
rows, cols = np.where(mask == 1)

print("Valid pixels:", len(rows))

Valid pixels: 42190905


In [ ]:
# RANDOM SAMPLE
idx = np.random.choice(len(rows), size=n_points, replace=False)

sample_rows = rows[idx]
sample_cols = cols[idx]

In [ ]:
# PIXEL -> POINTS
points = []

for r, c in zip(sample_rows, sample_cols):
    x, y = rasterio.transform.xy(transform, r, c)
    points.append(Point(x, y))

In [ ]:
# CREATE GDF
gdf = gpd.GeoDataFrame(
    {"label": [0] * len(points)},
    geometry=points,
    crs=crs
)

In [ ]:

gdf.to_file(output_path_random, driver="GPKG")

print("Saved:", output_path_random)

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: File /content/drive/MyDrive/cosesmic/layers/random/pseudi_absence_points_random has GPKG application_id, but non conformant file extension
  ogr_write(


Saved: /content/drive/MyDrive/cosesmic/layers/random/pseudi_absence_points_random


In [ ]:
# VALID PIXELS

rows, cols = np.where(mask == 1)

print("Valid pixels:", len(rows))

Valid pixels: 25485758


In [ ]:
# LOAD POLYGON

gdf_poly = gpd.read_file(polygon_path)



In [ ]:
points_gdf = gdf_poly.sample_points(size=n_points)

KeyboardInterrupt: 

In [ ]:
points_gdf = points_gdf.explode(ignore_index=True)

In [ ]:
points_gdf = points_gdf.iloc[:n_points].copy()

In [ ]:
points_gdf["label"] = 0

In [ ]:
points_gdf.to_file(output_path, driver="GPKG")

print("Generated:", len(points_gdf))
print("Saved:", output_path)

In [ ]:
# # merge polygons into one geometry
# polygon = gdf_poly.geometry.union_all()

# # bounds
# minx, miny, maxx, maxy = polygon.bounds

In [ ]:
# RANDOM POINT GENERATION
points = []

while len(points) < n_points:

    x = np.random.uniform(minx, maxx)
    y = np.random.uniform(miny, maxy)

    p = Point(x, y)

    if polygon.contains(p):
        points.append(p)

print("Generated points:", len(points))

In [ ]:
# CREATE GEODATAFRAME
points_gdf = gpd.GeoDataFrame(
    {"label": [0] * len(points)},
    geometry=points,
    crs=gdf_poly.crs
)


In [ ]:
points_gdf.to_file(output_path, driver="GPKG")

print("Saved:", output_path)